# Hyper-parameters Tuning

## Load Text

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', 500)

df = pd.read_csv("t_combined_nodup_13-23_compNameOK_retainedNoEmpty_namesANDcomps_removedFull_w_scores.csv", low_memory=False)
df['isDem'] = 0
df.loc[df['averageDem']>=0.5,'isDem']=1
df['isRep'] = 0
df.loc[df['averageDem']<0.5,'isRep']=1
df['isDemOrRep'] = df['isDem'] + df['isRep']

docs = df['componenttext_masked'].tolist()
print(len(docs))

df_ideo = df.dropna(subset='averageDem')
docs_ideo = df_ideo['componenttext_masked'].tolist()
print(df_ideo.shape)

In [ ]:
df['transcriptid'].nunique()

In [ ]:
df.head(-5)

## Load Model and Tune Paras

In [ ]:
# Load Embedding Model

from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("./all-mpnet-base-v2")

In [ ]:
# Calculate Embedding

embeddings = embedding_model.encode(docs_ideo, show_progress_bar=True)

In [ ]:
embeddings.shape

In [ ]:
import numpy as np

np.savez('embeddings_mpnet-base-v2_masked_13-23_2026Jul3.npz', embeddings=embeddings)

In [ ]:
# Hyperparameters

from hdbscan import HDBSCAN
from umap import UMAP
from sklearn.feature_extraction.text import CountVectorizer


umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)

hdbscan_model = HDBSCAN(min_cluster_size=120, min_samples=100, metric='euclidean', cluster_selection_method='eom', prediction_data=True)

vectorizer_model = CountVectorizer(stop_words="english", min_df=10, ngram_range=(1, 2))

In [ ]:
# Representation

#import openai
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance, PartOfSpeech

# KeyBERT
keybert_model = KeyBERTInspired()

# Part-of-Speech
#pos_model = PartOfSpeech("en_core_web_sm")

# MMR
mmr_model = MaximalMarginalRelevance(diversity=0.3)


# All representation models
representation_model = {
    "KeyBERT": keybert_model,
    "MMR": mmr_model,
   # "POS": pos_model
}

In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(

  # Pipeline models
  embedding_model=embedding_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
  representation_model=representation_model,

  # Hyperparameters
  top_n_words=15,
  verbose=True
)

topics, probs = topic_model.fit_transform(docs_ideo, embeddings)

In [ ]:
topic_model.get_topic_info()

In [ ]:
topic_model.get_topic_info().to_excel("./bertopic_ideo_13-23_relaxed_354Topics/topic_info_354Topics.xlsx")

In [ ]:
# Save Model
topic_model.save("bertopic_ideo_13-23_relaxed_354Topics", serialization="safetensors", save_ctfidf=True, save_embedding_model=embedding_model)

In [ ]:
# Get topic distribution (Fast)

topic_distr, _ = topic_model.approximate_distribution(docs_ideo, batch_size=1000, use_embedding_model=False)
print(topic_distr.shape)

In [ ]:
# Get topic distribution

topic_distr_slow, _ = topic_model.approximate_distribution(docs_ideo, batch_size=1000, use_embedding_model=True)
print(topic_distr.shape)

In [ ]:
# Save Topic distributions
import os
import numpy as np

# Define the folder and file paths
folder_path = './classification_input/13-23_relaxed_354Topics'
file_path = os.path.join(folder_path, 'topic_distr.npz')

# Create the folder if it does not exist
os.makedirs(folder_path, exist_ok=True)

# Save the topic distribution in a compressed file (.npz)
np.savez_compressed(file_path, topic_distr=topic_distr)

# Calc Document Topics (For all transcripts, including those without ideology labels)

In [ ]:
# Load Transcripts
import pandas as pd
pd.set_option('display.max_columns', 500)

df = pd.read_csv("t_combined_nodup_13-23_compNameOK_retainedNoEmpty_namesANDcomps_removedFull_w_scores.csv", low_memory=False, usecols=['transcriptcomponentid','componenttext_masked'])
docs = df['componenttext_masked'].tolist()
print(len(docs))

In [ ]:
# Load Embedding Model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("./all-mpnet-base-v2")

In [ ]:
# Load BERTopic Model
from bertopic import BERTopic

topic_model = BERTopic.load("./bertopic_ideo_13-23_relaxed_354Topics", embedding_model=embedding_model)

In [ ]:
# Calculate Embedding (可以不做这一步,因为计算topic的时候可以不用embedding model)

# embeddings = embedding_model.encode(docs, show_progress_bar=True)

In [ ]:
# Get topic distribution (Fast)

topic_distr, _ = topic_model.approximate_distribution(docs, batch_size=1000, use_embedding_model=False)
print(topic_distr.shape)

In [ ]:
# Save Topic distributions
import os
import numpy as np

# Define the folder and file paths
folder_path = './classification_input/13-23_relaxed_354Topics'
file_path = os.path.join(folder_path, 'topic_distr_Full_wNoIdeoLabel.npz')

# Create the folder if it does not exist
os.makedirs(folder_path, exist_ok=True)

# Save the topic distribution in a compressed file (.npz)
np.savez_compressed(file_path, topic_distr=topic_distr)

In [ ]:
topic_distr.shape

# Calc Document Topics (Contrast Set)

In [ ]:
# Load Transcripts
import pandas as pd
pd.set_option('display.max_columns', 500)

df = pd.read_csv("complete_families_all_batches.csv", low_memory=False, usecols=['Transcript ID','Component ID(s)','T00 Original','T10 Construct Rewrite','T01 Context Rewrite'])
docs_T00 = df['T00 Original'].tolist()
docs_T10 = df['T10 Construct Rewrite'].tolist()
docs_T01 = df['T01 Context Rewrite'].tolist()
print(len(docs_T00))
print(len(docs_T01))
print(len(docs_T10))

In [ ]:
# Load Embedding Model
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("./all-mpnet-base-v2")

In [ ]:
# Load BERTopic Model
from bertopic import BERTopic

topic_model = BERTopic.load("./bertopic_ideo_13-23_relaxed_354Topics", embedding_model=embedding_model)

In [ ]:
# Calculate Embedding (可以不做这一步,因为计算topic的时候可以不用embedding model)

# embeddings = embedding_model.encode(docs, show_progress_bar=True)

In [ ]:
# Get topic distribution (Fast)

topic_distr_T00, _ = topic_model.approximate_distribution(docs_T00, use_embedding_model=False)
print(topic_distr_T00.shape)

topic_distr_T01, _ = topic_model.approximate_distribution(docs_T01, use_embedding_model=False)
print(topic_distr_T01.shape)

topic_distr_T10, _ = topic_model.approximate_distribution(docs_T10, use_embedding_model=False)
print(topic_distr_T10.shape)

In [ ]:
df

In [ ]:
topic_distr_T00.shape

In [ ]:
# Save Topic distributions
import os
import numpy as np

np.savez_compressed("./classification_input/13-23_relaxed_354Topics/topic_distr_Full_wNoIdeoLabel_T00.npz", topic_distr=topic_distr_T00)
np.savez_compressed("./classification_input/13-23_relaxed_354Topics/topic_distr_Full_wNoIdeoLabel_T01.npz", topic_distr=topic_distr_T01)
np.savez_compressed("./classification_input/13-23_relaxed_354Topics/topic_distr_Full_wNoIdeoLabel_T10.npz", topic_distr=topic_distr_T10)

# Get 25 Words

In [ ]:
# Load Embedding Model
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("./all-mpnet-base-v2")

# Load BERTopic Model
from bertopic import BERTopic
topic_model = BERTopic.load("./bertopic_ideo_13-23_relaxed_354Topics", embedding_model=embedding_model)

In [ ]:
import numpy as np
import pandas as pd

# 全部词汇
vocab = np.asarray(
    topic_model.vectorizer_model.get_feature_names_out()
)

# c-TF-IDF 矩阵中的行与排序后的 Topic ID 对应
topic_ids = sorted(topic_model.topic_representations_.keys())

assert topic_model.c_tf_idf_.shape[0] == len(topic_ids)

records = []

for row_index, topic_id in enumerate(topic_ids):
    row = topic_model.c_tf_idf_.getrow(row_index)

    # 按 c-TF-IDF 分数从高到低排序
    positions = np.argsort(row.data)[::-1][:25]

    for rank, position in enumerate(positions, start=1):
        records.append({
            "Topic": topic_id,
            "Rank": rank,
            "Word": vocab[row.indices[position]],
            "Score": row.data[position]
        })

top25_long = pd.DataFrame(records)

# 每个 Topic 一行，word_1 到 word_25
top25_wide = (
    top25_long
    .pivot(index="Topic", columns="Rank", values="Word")
    .rename(columns=lambda rank: f"word_{rank}")
    .reset_index()
)

display(top25_wide)

In [ ]:
output_path = (
    "./bertopic_ideo_13-23_relaxed_354Topics/"
    "topic_top25_words.xlsx"
)

with pd.ExcelWriter(output_path) as writer:
    top25_wide.to_excel(
        writer,
        sheet_name="Top25_Wide",
        index=False
    )
    top25_long.to_excel(
        writer,
        sheet_name="Top25_With_Scores",
        index=False
    )

print(output_path)